# Data Understanding
## Data Collection
Langkah pertama dalam proyek ini adalah mengumpulkan data polutan udara (seperti NO₂, CO dan SO₂) yang bertipe deret waktu (_Time Series_). Dataset ini diambil dari platform satelit [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/).

Buat akun terlebih dahulu di website Copernicus agar bisa melakukan crawling data menggunakan library openEO.

### Install Library

Untuk melakukan proses crawling data, kita membutuhkan pustaka Python pendukung yaitu `openeo` untuk berkomunikasi dengan API Copernicus.

```bash
pip install openeo
```

### Autentikasi dan Pengambilan Data

Skrip di bawah ini melakukan proses autentikasi untuk menghubungkan sistem lokal kita dengan server Copernicus menggunakan _device code flow_.

```python
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
```

Saat menjalankan baris di atas, akan muncul permintaan autentikasi:

```
Visit (link authentikasi) 📋 to authenticate.
✅ Authorized successfully
Authenticated using device code flow.
```

Klik link autentikasi lalu login menggunakan akun Copernicus.

### Definisi Area dan Pengambilan Data NO₂, SO₂ dan CO dari geojson

Setelah berhasil masuk, langkah selanjutnya adalah menentukan wilayah spesifik. Titik koordinat batas wilayah Tuban (Poligon) didapatkan menggunakan alat bantu pemetaan [geojson.io](https://geojson.io) dengan menggambar kotak di atas wilayah yang diinginkan kemudian menyalin koordinatnya.

![Grafik Data](../../img/polutan/tuban.png)

Koordinat yang didapatkan dimasukkan ke dalam variabel `aoi` (Area of Interest). Satelit Sentinel-5P kemudian diminta untuk mengambil data polutan berdasarkan _bounding box_ wilayah tersebut dengan menyesuaikan variabel `s5post` atribut `bands`. 

Karena satelit mungkin merekam area yang sama beberapa kali, dilakukan **agregasi temporal harian** agar hanya terdapat rata-rata satu data per hari. Dilanjutkan dengan **agregasi spasial** agar seluruh _grid_ pada wilayah Tuban dirata-rata menjadi satu nilai tunggal.

```python
aoi = {
    "type": "Polygon",
   
    "coordinates": [
        [
            [
              111.66191298444176,
              -6.737450756719085
            ],
            [
              112.16520539429672,
              -6.737450756719085
            ],
            [
              112.16520539429672,
              -7.125663462269856
            ],
            [
              111.66191298444176,
              -7.125663462269856
            ],
            [
              111.66191298444176,
              -6.737450756719085
            ]
        ]
    ]
}
s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-25", "2026-08-25"],
    spatial_extent={
        "west": 111.66191298444176,
        "south": -7.125663462269856,
        "east": 112.16520539429672,
        "north": -6.737450756719085
    },
    # Disesuaikan dengan data yang dibutuhkan
    bands=["NO2"],
)

# Agregasi harian agar tidak ada lebih dari satu data per hari
s5p_no2_daily = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Agregasi spasial untuk menghasilkan rata-rata time series per AOI
s5p_no2_aoi = s5p_no2_daily.aggregate_spatial(reducer="mean", geometries=aoi)

# Simpan hasil sebagai CSV
result = s5p_no2_aoi.save_result(format="CSV")

# Jalankan job
job = result.create_job(title="s5p_no2_timeseries")
job.start_and_wait()

# Download
job.get_results().download_files("output_no2")
```

Tunggu proses selesai. Status dan progres eksekusi bisa dipantau di [openEO editor](https://editor.openeo.org/?server=https%3A%2F%2Fopeneo.dataspace.copernicus.eu%2Fopeneo%2F1.2). Setelah diproses oleh server, output akan otomatis diunduh dalam format **CSV**.

![Grafik Data](../../img/polutan/editor.png)

```
0:00:00 Job 'j-2608250945264132925ebef4140e0037': send 'start'
0:00:03 Job 'j-2608250945264132925ebef4140e0037': queued (progress 0%)
0:00:08 Job 'j-2608250945264132925ebef4140e0037': queued (progress 0%)
0:00:15 Job 'j-2608250945264132925ebef4140e0037': queued (progress 0%)
0:00:23 Job 'j-2608250945264132925ebef4140e0037': queued (progress 0%)
0:00:33 Job 'j-2608250945264132925ebef4140e0037': queued (progress 0%)
0:00:46 Job 'j-2608250945264132925ebef4140e0037': running (progress N/A)
0:01:02 Job 'j-2608250945264132925ebef4140e0037': running (progress N/A)
0:01:21 Job 'j-2608250945264132925ebef4140e0037': running (progress N/A)
0:01:45 Job 'j-2608250945264132925ebef4140e0037': running (progress N/A)
0:02:16 Job 'j-2608250945264132925ebef4140e0037': running (progress N/A)
0:02:53 Job 'j-2608250945264132925ebef4140e0037': running (progress N/A)
0:03:40 Job 'j-2608250945264132925ebef4140e0037': finished (progress 100%)
```

### Hasil CSV
Pada tahap terakhir, kita memuat file CSV (SO₂, CO dan NO₂) yang telah dirapikan menggunakan pustaka Pandas. Data ini sekarang sudah terstruktur sebagai dataset _Time Series_ dan siap digunakan untuk analisis lanjutan. Berikut adalah cuplikan data tersebut:

1. CO

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/polutan/CO.csv")
df.head(5)

,date,feature_index,CO
0,2025-09-25T00:00:00.000Z,0,0.031352
1,2025-09-27T00:00:00.000Z,0,0.034081
2,2025-09-23T00:00:00.000Z,0,0.034661
3,2025-09-24T00:00:00.000Z,0,0.037663
4,2025-09-21T00:00:00.000Z,0,0.031104


2. SO2

In [2]:
df = pd.read_csv("../../data/polutan/SO2.csv")
df.head(5)

,date,feature_index,SO2
0,2025-08-25T00:00:00.000Z,0,0.000253
1,2025-08-26T00:00:00.000Z,0,0.000262
2,2025-08-27T00:00:00.000Z,0,0.000100
3,2025-08-24T00:00:00.000Z,0,NaN
4,2025-11-10T00:00:00.000Z,0,0.000105


3. NO 2

In [3]:
df = pd.read_csv("../../data/polutan/NO2.csv")
df.head(5)

,date,feature_index,NO2
0,2026-07-21T00:00:00.000Z,0,0.000037
1,2026-07-15T00:00:00.000Z,0,0.000044
2,2026-07-20T00:00:00.000Z,0,0.000047
3,2026-07-17T00:00:00.000Z,0,0.000040
4,2026-07-16T00:00:00.000Z,0,0.000040


### Normalisasi Tanggal

Data waktu (tanggal) yang diperoleh dari Copernicus menyertakan zona waktu yang tidak diperlukan. Oleh karena itu, kita perlu menormalisasinya menjadi format standar yang seragam yaitu `YYYY-MM-DD` agar lebih mudah diolah. Berikut adalah kode yang digunakan untuk menyeragamkan format tanggal:
```python
import pandas as pd

df = pd.read_csv("SO2.csv")

# pastikan kolom tanggal valid
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# ambil hanya bulan dan tahun
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

new_df = pd.DataFrame({
    "date": df['date'],
    "SO2": df['SO2']
})

new_df.to_csv("SO2_Timeseries.csv", index=False)
```
Setelah proses normalisasi dilakukan pada seluruh dataset polutan, format waktu pada dataset menjadi lebih rapi dan konsisten. Berikut adalah cuplikan dataset setelah tanggal dinormalisasi:


1. CO

In [4]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/polutan/CO_Timeseries.csv")
df.head(5)

,date,CO
0,2025-09-25,0.031352
1,2025-09-27,0.034081
2,2025-09-23,0.034661
3,2025-09-24,0.037663
4,2025-09-21,0.031104


2. SO2

In [5]:
df = pd.read_csv("../../data/polutan/SO2_Timeseries.csv")
df.head(5)

,date,SO2
0,2025-08-25,0.000253
1,2025-08-26,0.000262
2,2025-08-27,0.000100
3,2025-08-24,NaN
4,2025-11-10,0.000105


3. NO2

In [6]:
df = pd.read_csv("../../data/polutan/NO2_Timeseries.csv")
df.head(5)

,date,NO2
0,2026-07-21,0.000037
1,2026-07-15,0.000044
2,2026-07-20,0.000047
3,2026-07-17,0.000040
4,2026-07-16,0.000040


## Missing Values

_Missing values_ (nilai yang hilang) adalah kondisi di mana terdapat informasi yang kosong atau tidak terekam dalam dataset. Pada kasus data deret waktu yang diambil menggunakan satelit, kekosongan data ini wajar terjadi, biasanya akibat faktor cuaca (area tertutup awan tebal sehingga sensor tidak dapat membaca permukaan bumi) atau karena orbit satelit yang tidak merekam area tersebut pada hari tertentu. Mengidentifikasi keberadaan _missing values_ sangat penting sebelum melakukan analisis lebih lanjut.

Pada proyek ini, kita mengecek dua bentuk _missing values_:
1. **Tanggal yang Hilang**: Memastikan apakah ada urutan hari yang terlewat (bolong) dari rentang waktu awal hingga akhir (25 Agustus 2025 - 25 Agustus 2026).
2. **Data yang Hilang**: Memeriksa jumlah nilai polutan yang kosong (`NaN`) pada record tanggal yang sudah terekam.

### Tanggal Yang Hilang
1. CO

In [7]:
import pandas as pd

df = pd.read_csv("../../data/polutan/CO_Timeseries.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')


2. SO2

In [8]:
import pandas as pd

df = pd.read_csv("../../data/polutan/SO2_Timeseries.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')


3. NO₂

In [9]:
import pandas as pd

df = pd.read_csv("../../data/polutan/NO2_Timeseries.csv")
df['date'] = pd.to_datetime(df['date'])

# Buat rentang tanggal lengkap
start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Cek tanggal yang hilang
missing_dates = full_range.difference(df['date'])

print(f"Jumlah hari missing: {len(missing_dates)}")
print("Daftar tanggal missing:")
print(missing_dates)

Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')


### Data Yang Hilang

Selain urutan tanggal, kita juga mengecek jumlah baris data yang memiliki nilai konsentrasi polutan kosong (`NaN`).

1. CO

In [10]:
df = pd.read_csv("../../data/polutan/CO_Timeseries.csv")
missing_value = df['CO'].isna().sum()
print(missing_value)

81


Implementasi pada tools `Orange Data Mining`
```{image} ../../img/polutan/co_missing.png
:alt: Grafik Data
:width: 100%
:align: center
```

2. SO₂

In [11]:
df = pd.read_csv("../../data/polutan/SO2_Timeseries.csv")
missing_value = df['SO2'].isna().sum()
print(missing_value)

40


Implementasi pada tools `Orange Data Mining`

```{image} ../../img/polutan/so2_missing.png
:alt: Grafik Data
:width: 100%
:align: center
```

3. NO₂

In [12]:
df = pd.read_csv("../../data/polutan/NO2_Timeseries.csv")
missing_value = df['NO2'].isna().sum()
print(missing_value)

66


Implementasi pada tools `Orange Data Mining`

```{image} ../../img/polutan/no2_missing.png
:alt: Grafik Data
:width: 100%
:align: center
```



## Outliers

_Outliers_ (pencilan) adalah titik data yang nilainya menyimpang secara drastis atau ekstrem dari mayoritas distribusi data lainnya. Pada data deret waktu kualitas udara, _outlier_ bisa jadi merupakan lonjakan polusi nyata yang terjadi akibat peristiwa tertentu (misalnya kebakaran hutan atau peningkatan aktivitas industri mendadak), atau bisa juga sekadar _noise_ / _error_ pada pembacaan sensor satelit.

Pada tahap _data understanding_ ini, kita mengeksplorasi _outliers_ menggunakan algoritma **Isolation Forest** dari pustaka `scikit-learn`. Algoritma deteksi anomali ini bekerja dengan cara "mengisolasi" observasi melalui pemisahan data secara acak, di mana anomali akan lebih cepat/mudah diisolasi. Kita mengatur parameter _contamination_ (estimasi persentase _outlier_ di dalam dataset) sebesar 5%. Hasil prediksi dari model yang bernilai `-1` menandakan bahwa baris tersebut terdeteksi sebagai _outlier_.

1. CO

In [13]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../../data/polutan/CO_Timeseries.csv")
df_clean = df.dropna(subset=['CO']).copy()

model = IsolationForest(contamination=0.05, random_state=42) # contamination 0.05 = 5%
pred = model.fit_predict(df_clean[['CO']])

# Nilai -1 merepresentasikan outlier
jumlah_outlier = (pred == -1).sum()
print("Jumlah outlier:", jumlah_outlier)

Jumlah outlier: 15


Implementasi pada tools `Orange Data Mining`

```{image} ../../img/polutan/co_outliers.png
:alt: Grafik Data
:width: 100%
:align: center
:class: mabot-gambar
```

```{image} ../../img/polutan/sp_co.png
:alt: Grafik Data
:width: 100%
:align: center
```

2. SO₂

In [14]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../../data/polutan/SO2_Timeseries.csv")
df_clean = df.dropna(subset=['SO2']).copy()

model = IsolationForest(contamination=0.05, random_state=42) # contamination 0.05 = 5%
pred = model.fit_predict(df_clean[['SO2']])

# Nilai -1 merepresentasikan outlier
jumlah_outlier = (pred == -1).sum()
print("Jumlah outlier:", jumlah_outlier)

Jumlah outlier: 17


Implementasi pada tools `Orange Data Mining`

```{image} ../../img/polutan/so2_outliers.png
:alt: Grafik Data
:width: 100%
:align: center
:class: mabot-gambar
```

```{image} ../../img/polutan/sp_so2.png
:alt: Grafik Data
:width: 100%
:align: center
```

3. NO₂

In [15]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../../data/polutan/NO2_Timeseries.csv")
df_clean = df.dropna(subset=['NO2']).copy()

model = IsolationForest(contamination=0.05, random_state=42) # contamination 0.05 = 5%
pred = model.fit_predict(df_clean[['NO2']])

# Nilai -1 merepresentasikan outlier
jumlah_outlier = (pred == -1).sum()
print("Jumlah outlier:", jumlah_outlier)

Jumlah outlier: 15


Implementasi pada tools `Orange Data Mining`

```{image} ../../img/polutan/no2_outliers.png
:alt: Grafik Data
:width: 100%
:align: center
:class: mabot-gambar
```

```{image} ../../img/polutan/sp_no2.png
:alt: Grafik Data
:width: 100%
:align: center
```


## Menggabungkan File CSV

Setelah setiap dataset polutan (CO, NO₂, dan SO₂) dinormalisasi dan dianalisis nilai kosong serta pencilan (outliers)-nya, langkah selanjutnya adalah menggabungkan keempat file tersebut menjadi satu dataset terpadu. Karena keempat data tersebut direkam dengan rentang waktu harian yang sama, kita dapat menggabungkannya berdasarkan kolom tanggal (`date`). Penggabungan ini akan mempermudah proses analisis multivariat dan pemodelan pada tahap selanjutnya, karena seluruh fitur parameter polutan udara kini berada dalam satu tabel yang terpusat.

Berikut adalah kode Python menggunakan pustaka Pandas untuk menyatukan keempat dataset tersebut dan menyimpannya ke dalam file baru bernama `Polutan_Tuban.csv`:
```python
import pandas as pd

df_co = pd.read_csv("CO_Timeseries.csv")
df_no2 = pd.read_csv("NO2_Timeseries.csv")
df_so2 = pd.read_csv("SO2_Timeseries.csv")

dataframe_merged = pd.DataFrame({
    "date": df_co['date'],
    "CO": df_co['CO'],
    "NO2": df_no2['NO2'],
    "SO2": df_so2['SO2']
})

dataframe_merged.to_csv("Polutan_Tuban.csv", index=False)
```

In [16]:
df = pd.read_csv("../../data/polutan/Polutan_Tuban.csv")
df.head(5)

,date,CO,NO2,SO2
0,2025-09-25,0.031352,0.000037,0.000253
1,2025-09-27,0.034081,0.000044,0.000262
2,2025-09-23,0.034661,0.000047,0.000100
3,2025-09-24,0.037663,0.000040,NaN
4,2025-09-21,0.031104,0.000040,0.000105
